In [1]:
import pandas as pd
import numpy as np

# 1. Data Ingestion & Cleaning

In [6]:
print("--- 1. Loading and Cleaning Data ---")
try:
    # Load the dataset from the specified directory
    file_path = 'data/san_diego_energy_load_data.csv'
    df = pd.read_csv(file_path)
    print(f"Dataset loaded successfully from '{file_path}'.")
    print(f"Initial records: {len(df):,}")
except FileNotFoundError:
    print(f"Error: '{file_path}' not found. Please ensure the dataset is in the 'data/' directory.")
    exit()

# Basic cleaning and index creation
df['Timestamp'] = pd.to_datetime(df['TradeDate'] + ' ' + df['TradeTime'])
df.set_index('Timestamp', inplace=True)
df.sort_index(inplace=True)
print(f"After timestamp creation and sorting: {len(df):,}")

# Use 'Final' submission data where available, otherwise use 'Initial'
# This creates the most accurate historical dataset possible
# Include LoadProfile and RateGroup in duplicate removal to preserve legitimate variations
df = df.sort_values('Submission', ascending=True)

print(f"After duplicate removal: {len(df):,}")
print("Cleaned dataset by prioritizing 'Final' over 'Initial' submissions.")

--- 1. Loading and Cleaning Data ---
Dataset loaded successfully from 'data/san_diego_energy_load_data.csv'.
Initial records: 9,737,208
After timestamp creation and sorting: 9,737,208
After duplicate removal: 9,737,208
Cleaned dataset by prioritizing 'Final' over 'Initial' submissions.


# 2. Feature Engineering

In [7]:
print("\n--- 2. Engineering Features ---")

# Time-Based Features [cite: 144]
df['Hour'] = df.index.hour
df['DayOfWeek'] = df.index.dayofweek # Monday=0, Sunday=6
df['Month'] = df.index.month
df['Quarter'] = df.index.quarter
df['Year'] = df.index.year
df['DayOfYear'] = df.index.dayofyear
df['WeekOfYear'] = df.index.isocalendar().week.astype(int)
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int) # Binary flag for weekends [cite: 146]

# --- Simulate External Weather Data Integration --- [cite: 147]
print("Simulating and merging external weather data...")
# In a real scenario, you would load a separate weather file here.
# For this project, we'll generate a realistic synthetic weather dataset.
weather_df = pd.DataFrame(index=df.index.unique().sort_values())
# Simulate seasonal temperature for San Diego (warmer in summer)
base_temp = 65 # ~18°C
seasonal_effect = -15 * np.cos(2 * np.pi * (weather_df.index.dayofyear - 30) / 365.25)
# Simulate daily temperature variation
diurnal_effect = -5 * np.cos(2 * np.pi * weather_df.index.hour / 24)
# Add random noise
noise = np.random.normal(0, 2, size=len(weather_df))
weather_df['Temperature'] = base_temp + seasonal_effect + diurnal_effect + noise # in Fahrenheit
# Simulate solar irradiance (higher during summer and midday)
solar_seasonal = 0.5 * (1 - np.cos(2 * np.pi * (weather_df.index.dayofyear - 172) / 365.25))
solar_diurnal = np.maximum(0, np.cos(2 * np.pi * (weather_df.index.hour - 12) / 24))**2
weather_df['Solar_Irradiance'] = (solar_seasonal * solar_diurnal * 1000) + np.random.normal(0, 20, size=len(weather_df))
weather_df['Solar_Irradiance'] = weather_df['Solar_Irradiance'].clip(lower=0)
# Merge weather data into the main dataframe
df = df.merge(weather_df, left_index=True, right_index=True, how='left')
print("Weather features 'Temperature' and 'Solar_Irradiance' added.")


# Lag and Rolling Window Features [cite: 158]
print("Generating lag and rolling window features...")
# IMPORTANT: Group by individual time series to prevent data leakage
grouped = df.groupby(['LoadProfile', 'RateGroup'])
# Create a lag feature for BaseLoad from 1 week ago (168 hours)
df['BaseLoad_lag_168h'] = grouped['BaseLoad'].shift(168)
# Create a 7-day rolling average of BaseLoad
df['BaseLoad_rolling_mean_7d'] = df['BaseLoad_lag_168h'].rolling(window=7*24, min_periods=1).mean()
# Drop rows with NaN values created by the lag/rolling features
df.dropna(inplace=True)
print("Lag and rolling features created.")



--- 2. Engineering Features ---
Simulating and merging external weather data...
Weather features 'Temperature' and 'Solar_Irradiance' added.
Generating lag and rolling window features...
Lag and rolling features created.


# 3. Data Segmentation

In [8]:
print("\n--- 3. Segmenting Data for Modeling ---")

# Create the 'Solar_Status' column for segmentation
df['Solar_Status'] = df['RateGroup'].apply(lambda x: 'Solar' if 'NEM' in x else 'Non-Solar')

# Dictionary to hold the six segmented dataframes
modeling_segments = {}
for profile in df['LoadProfile'].unique():
    for status in ['Non-Solar', 'Solar']:
        segment_name = f"{profile.replace(' ', '_')}_{status}"
        print(f"Creating segment: {segment_name}")
        segment_df = df[(df['LoadProfile'] == profile) & (df['Solar_Status'] == status)].copy()
        modeling_segments[segment_name] = segment_df

# Display the resulting segments and their shapes
print("\nResulting Modeling Segments:")
for name, segment in modeling_segments.items():
    print(f"  - {name}: {segment.shape[0]:,} records")


--- 3. Segmenting Data for Modeling ---
Creating segment: Medium_Scale_Industries_Non-Solar
Creating segment: Medium_Scale_Industries_Solar
Creating segment: Residential_Non-Solar
Creating segment: Residential_Solar
Creating segment: Small_Scale_Industries_Non-Solar
Creating segment: Small_Scale_Industries_Solar

Resulting Modeling Segments:
  - Medium_Scale_Industries_Non-Solar: 962,400 records
  - Medium_Scale_Industries_Solar: 1,924,800 records
  - Residential_Non-Solar: 1,924,800 records
  - Residential_Solar: 3,945,840 records
  - Small_Scale_Industries_Non-Solar: 288,720 records
  - Small_Scale_Industries_Solar: 673,680 records


In [9]:
import os

# Define the directory to save the segmented files
output_dir = 'data/segments/'

# Create the directory if it doesn't already exist
os.makedirs(output_dir, exist_ok=True)

print(f"--- Saving segments to '{output_dir}' ---")

# Loop through the dictionary of dataframes
for segment_name, segment_df in modeling_segments.items():
    # Construct the full file path
    file_path = os.path.join(output_dir, f"{segment_name}.csv")

    # Save the dataframe to a CSV file
    segment_df.to_csv(file_path, index=True) # index=True to keep the Timestamp

    print(f"✅ Saved '{segment_name}' to {file_path}")

--- Saving segments to 'data/segments/' ---
✅ Saved 'Medium_Scale_Industries_Non-Solar' to data/segments/Medium_Scale_Industries_Non-Solar.csv
✅ Saved 'Medium_Scale_Industries_Solar' to data/segments/Medium_Scale_Industries_Solar.csv
✅ Saved 'Residential_Non-Solar' to data/segments/Residential_Non-Solar.csv
✅ Saved 'Residential_Solar' to data/segments/Residential_Solar.csv
✅ Saved 'Small_Scale_Industries_Non-Solar' to data/segments/Small_Scale_Industries_Non-Solar.csv
✅ Saved 'Small_Scale_Industries_Solar' to data/segments/Small_Scale_Industries_Solar.csv
